# Clase 09 — Ejercicio intergrador Clínica MediCare

> Notebook de práctica. Ejecutá las celdas a medida que avanzás en el artículo.

La clínica MediCare lleva varios años registrando turnos médicos en un sistema interno. El sistema fue cambiando de versión con el tiempo, y los datos se fueron exportando a una base de datos SQLite sin demasiado control de calidad.

El director médico necesita un informe con algunas preguntas concretas sobre el funcionamiento de la clínica, pero antes de poder responderlas, alguien tiene que hacer la limpieza. Ese trabajo es el de ustedes, no se les dice qué problemas hay en los datos. Parte del ejercicio es encontrarlos.

> El archivo de la base de datos `medicare.db` ya está disponible para que lo uses en este ejercicio.

### 1) Exploración: ¿qué tiene esta base de datos?

Antes de tocar nada, el primer trabajo es entender qué hay adentro. Conectense a `medicare.db` y respondan estas preguntas.

1. ¿Qué tablas tiene la base de datos?
2. ¿Qué columnas y tipos tiene cada tabla?
3. ¿Cuántas filas tiene cada tabla?

In [2]:
pip install pandas

  Using cached pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
Using cached numpy-2.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [numpy]  WARNING: The scripts f2py and numpy-config are installed in '/usr/local/python/3.12.1/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import sqlite3
import pandas as pd

# Conexión a la base de datos
conexion = sqlite3.connect("medicare.db")

In [5]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

tablas = pd.read_sql(query, conexion)
print(tablas)

        name
0  pacientes
1    medicos
2     turnos


In [10]:
import sqlite3

conexion = sqlite3.connect("medicare.db")
cursor = conexion.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tablas = cursor.fetchall()

print(tablas)

[('pacientes',), ('medicos',), ('turnos',)]


In [11]:
for tabla in tablas:
    nombre_tabla = tabla[0]
    print(f"\nTabla: {nombre_tabla}")

    cursor.execute(f"PRAGMA table_info({nombre_tabla});")
    columnas = cursor.fetchall()

    for columna in columnas:
        print(f"Columna: {columna[1]} - Tipo: {columna[2]}")




Tabla: pacientes
Columna: id - Tipo: INTEGER
Columna: nombre - Tipo: TEXT
Columna: dni - Tipo: TEXT
Columna: fecha_nacimiento - Tipo: TEXT
Columna: ciudad - Tipo: TEXT
Columna: obra_social - Tipo: TEXT

Tabla: medicos
Columna: id - Tipo: INTEGER
Columna: nombre - Tipo: TEXT
Columna: especialidad - Tipo: TEXT
Columna: matricula - Tipo: TEXT
Columna: activo - Tipo: TEXT

Tabla: turnos
Columna: id - Tipo: INTEGER
Columna: paciente_id - Tipo: INTEGER
Columna: medico_id - Tipo: INTEGER
Columna: fecha - Tipo: TEXT
Columna: hora - Tipo: TEXT
Columna: motivo - Tipo: TEXT
Columna: costo - Tipo: TEXT
Columna: estado - Tipo: TEXT


**Documenten sus hallazgos.** Antes de seguir al Paso 2, escriban en comentarios qué encontraron.

### 2) Diagnóstico: ¿qué problemas tienen los datos?

Traigan las tres tablas a DataFrames y hagan un diagnóstico completo de cada una. Para cada tabla respondan:

- ¿Cuántos nulos hay por columna? ¿Qué porcentaje representan?
- ¿Hay filas duplicadas? ¿Cuáles?
- ¿Hay columnas con tipos incorrectos que van a impedir operar con los datos?
- ¿Hay valores que parecen errores aunque no sean nulos? (Pista: revisen la columna `costo` en `turnos`)


In [12]:
import sqlite3
import pandas as pd

# Conexión
conexion = sqlite3.connect("medicare.db")

# Cargar las tablas en DataFrames
pacientes = pd.read_sql("SELECT * FROM pacientes", conexion)
medicos = pd.read_sql("SELECT * FROM medicos", conexion)
turnos = pd.read_sql("SELECT * FROM turnos", conexion)

conexion.close()

In [13]:
def diagnostico(df, nombre):
    print(f"\n===== {nombre.upper()} =====")

    # Cantidad de nulos
    print("\nNulos por columna:")
    print(df.isnull().sum())

    # Porcentaje de nulos
    print("\nPorcentaje de nulos:")
    print((df.isnull().mean() * 100).round(2))

    # Duplicados
    print("\nCantidad de filas duplicadas:")
    print(df.duplicated().sum())

    if df.duplicated().sum() > 0:
        print("\nFilas duplicadas:")
        print(df[df.duplicated()])

    # Tipos de datos
    print("\nTipos de datos:")
    print(df.dtypes)

In [14]:
diagnostico(pacientes, "Pacientes")
diagnostico(medicos, "Medicos")
diagnostico(turnos, "Turnos")


===== PACIENTES =====

Nulos por columna:
id                   0
nombre               0
dni                  7
fecha_nacimiento    11
ciudad               8
obra_social         12
dtype: int64

Porcentaje de nulos:
id                   0.00
nombre               0.00
dni                 10.61
fecha_nacimiento    16.67
ciudad              12.12
obra_social         18.18
dtype: float64

Cantidad de filas duplicadas:
0

Tipos de datos:
id                  int64
nombre                str
dni                   str
fecha_nacimiento      str
ciudad                str
obra_social           str
dtype: object

===== MEDICOS =====

Nulos por columna:
id              0
nombre          0
especialidad    0
matricula       2
activo          4
dtype: int64

Porcentaje de nulos:
id               0.0
nombre           0.0
especialidad     0.0
matricula       10.0
activo          20.0
dtype: float64

Cantidad de filas duplicadas:
0

Tipos de datos:
id              int64
nombre            str
especialidad 

In [15]:

print(turnos["costo"].unique())

<StringArray>
[ '3500',  '4200',  '8500',     nan,  '5500',  '-500',   'N/A',  '9200',
  '6800',      '', '-3500', '-1000']
Length: 12, dtype: str


In [16]:
turnos["costo_num"] = pd.to_numeric(turnos["costo"], errors="coerce")

print(turnos[turnos["costo_num"].isna()])

      id  paciente_id  medico_id       fecha   hora              motivo costo  \
5      6            6          6  2024-03-03  14:00     Fractura muñeca   NaN   
19    20            7          3  2024-03-10  10:00  Electrocardiograma   N/A   
21    22           55         17  2024-03-18  16:00     Chequeo general   N/A   
23    24            1         15  2024-03-09  16:00                 NaN   N/A   
38    39           64         18  2024-03-29  13:45   Estudio de rutina         
42    43           56         10  2024-03-03  14:45  Control de presión   NaN   
59    60           50          9  2024-04-25  10:45     Fractura muñeca   NaN   
69    70           27          1  2024-04-11  13:15    Esguince tobillo   N/A   
75    76           26         10  2024-03-17  08:45       Dolor rodilla   N/A   
81    82           66          2  2024-03-06  09:30                 NaN   N/A   
123  124           46          2  2024-04-02  14:00       Control anual   N/A   
159  160           41       

**Antes de seguir al Paso 3, escriban una lista con todos los problemas que encontraron, uno por línea.** No empiecen a limpiar todavía.

### 3) Decisiones: ¿qué hacemos con cada problema?

Este es el paso más importante del ejercicio. Para cada problema que encontraron en el Paso 2, decidan qué van a hacer y por qué. No hay una respuesta única correcta — lo que importa es que la decisión sea coherente con el análisis que necesitan hacer después.

Completen esta tabla antes de escribir código:

| Tabla | Columna | Problema | Decisión | Justificación |
|-------|---------|----------|----------|---------------|
| pacientes | ciudad | 2 nulos | ... | ... |
| pacientes | dni | 1 nulo | ... | ... |
| ... | ... | ... | ... | ... |


### 4) Limpieza: aplicar las decisiones

Ahora sí, implementen las decisiones del Paso 3. Recuerden:

- Primero eliminen duplicados
- Después manejen nulos
- Después conviertan tipos

In [20]:
import sqlite3
import pandas as pd

# Conectar a la base
conexion = sqlite3.connect("medicare.db")

# Cargar tablas
pacientes = pd.read_sql("SELECT * FROM pacientes", conexion)
medicos = pd.read_sql("SELECT * FROM medicos", conexion)
turnos = pd.read_sql("SELECT * FROM turnos", conexion)

conexion.close()

In [ ]:

pacientes = pacientes.drop_duplicates()
medicos = medicos.drop_duplicates()
turnos = turnos.drop_duplicates()

In [21]:
pacientes = pacientes.drop_duplicates()
medicos = medicos.drop_duplicates()
turnos = turnos.drop_duplicates()

pacientes["ciudad"] = pacientes["ciudad"].fillna("desconocida")p

pacientes["dni"] = pacientes["dni"].fillna("Sin DNI")

In [22]:
# Completar ciudad faltante
pacientes["ciudad"] = pacientes["ciudad"].fillna("Desconocida")

# Si DNI es identificador, podemos dejarlo como desconocido
pacientes["dni"] = pacientes["dni"].fillna("Sin DNI")

In [30]:
'''turnos["fecha"] = pd.to_datetime(
    turnos["fecha"],
    errors="coerce"
)'''

turnos["fecha"] = pd.to_datetime
turnos["fecha"],

errors= "coerce"


In [29]:
'''# Limpieza de pacientes
#df_pacientes_limpio = df_pacientes.copy()
df_pacientes_limpio = df_pacientes.copy()

# 1) Eliminar duplicados
#df_pacientes_limpio = df_pacientes_limpio.drop_duplicates()
df_pacientes_limpio["dni"] = df_pacientes_limpio["dni"].fillna("sin dato")

# 2) Manejar nulos
df_pacientes_limpio["ciudad"] = df_pacientes_limpio["ciudad"].fillna("Desconocida")
df_pacientes_limpio["dni"] = df_pacientes_limpio["dni"].fillna("Sin DNI")

# 3) Convertir tipos (ajustar según tus columnas)
# Ejemplo:
# df_pacientes_limpio["edad"] = pd.to_numeric(df_pacientes_limpio["edad"], errors="coerce")


# Limpieza de médicos
df_medicos_limpio = df_medicos.copy()

# 1) Eliminar duplicados
df_medicos_limpio = df_medicos_limpio.drop_duplicates()

# 2) Manejar nulos
df_medicos_limpio["especialidad"] = df_medicos_limpio["especialidad"].fillna("Sin especificar")

# 3) Convertir tipos (ejemplo si existe una columna fecha)
# df_medicos_limpio["fecha_ingreso"] = pd.to_datetime(
#     df_medicos_limpio["fecha_ingreso"], 
#     errors="coerce"
# )
''''''

# Limpieza de turnos
df_turnos_limpio = df_turnos.copy()

# 1) Eliminar duplicados
df_turnos_limpio = df_turnos_limpio.drop_duplicates()

# 2) Manejar nulos
# Completar costo incorrecto o faltante después de convertirlo
df_turnos_limpio["costo"] = pd.to_numeric(
    df_turnos_limpio["costo"],
    errors="coerce"
)

df_turnos_limpio["costo"] = df_turnos_limpio["costo"].fillna(
    df_turnos_limpio["costo"].median()
)

# 3) Convertir tipos
df_turnos_limpio["fecha"] = pd.to_datetime(
    df_turnos_limpio["fecha"],
    errors="coerce"
)

SyntaxError: incomplete input (3509091312.py, line 32)

In [ ]:
df_pacientes_limpio= df_pacientes.copy()

df.cost_es

NameError: name 'df_pacientes' is not defined

In [ ]:
# Limpieza de pacientes
df_pacientes_limpio = df_pacientes.copy()

# Tu código acá
df.cost_es

# Limpieza de médicos
df_medicos_limpio = df_medicos.copy()

# Tu código acá

# Limpieza de turnos
df_turnos_limpio = df_turnos.copy()
# Tu código acá

NameError: name 'df_pacientes' is not defined

> **Por qué `.copy()`:** cuando hacen `df_pacientes_limpio = df_pacientes`, no están creando una copia — están apuntando al mismo objeto. Cualquier cambio en `df_pacientes_limpio` modifica también `df_pacientes`. `.copy()` crea una copia independiente, así conservan los datos originales para comparar.

Al terminar, impriman un resumen de qué cambió en cada tabla:

In [ ]:
# tu codigo aquí
df:copu_list("sin dato").drop_list()
df.list_copy["con dato"]
print("solucion.md")
df-coment:cliente("numero"),

NameError: name 'copu_list' is not defined

### 5) — Análisis: responder las preguntas del director

Con los datos limpios, respondan estas cuatro preguntas. Para cada una, combinen SQL (para traer los datos necesarios) con Pandas (para calcular o presentar el resultado):

**Pregunta 1:** ¿Cuántos turnos realizó cada médico? Mostrá el resultado ordenado de mayor a menor.

**Pregunta 2:** ¿Cuál es el costo total facturado por especialidad médica? Considerá solo los turnos con estado `'realizado'`.

**Pregunta 3:** ¿Qué obra social tiene más pacientes registrados? ¿Y cuántos pacientes no tienen obra social?

**Pregunta 4:** ¿Cuántos turnos fueron cancelados o marcados como `'ausente'`? ¿Qué médico tuvo más cancelaciones?


### 6) Reporte final

Escriban un bloque de código que imprima un reporte con los resultados de los cuatro puntos anteriores, en formato legible. No es necesario que sea elaborado — lo importante es que quede claro qué encontraron.

In [ ]:
print("=" * 50)
print("REPORTE CLÍNICA MEDICARE")
print("=" * 50)

# tu reporte acá

→ [Ir a la solución](./solucion.md)